# OPMark_ref Marking Ratio Line Charts

Read metric files and draw one line chart per metric.

- `wsr.txt`: 1%, 2%, 3% marking ratio.
- `bleu.txt`: 0%, 1%, 2%, 3% marking ratio.
- `codebleu.txt`: 0%, 1%, 2%, 3% marking ratio.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'OPMark_ref_123%':
    BASE_DIR = Path('Draw/OPMark_ref_123%')

DEFAULT_MODELS = ['CodeBERT', 'CodeT5', 'StarCoder', 'QwenCoder']
COLORS = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']
MARKERS = ['o', 's', '^', 'D']


In [ ]:
def load_metric_file(path):
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        return [], []

    model_names = []
    groups = []
    for block in text.split('\n\n'):
        lines = [line.strip() for line in block.splitlines() if line.strip()]
        if not lines:
            continue

        try:
            values = [float(line) for line in lines]
            model_name = DEFAULT_MODELS[len(groups)]
        except ValueError:
            model_name = lines[0]
            values = [float(line) for line in lines[1:]]

        model_names.append(model_name)
        groups.append(values)

    if len(model_names) != len(DEFAULT_MODELS):
        raise ValueError(f'{path}: expected {len(DEFAULT_MODELS)} model groups, got {len(model_names)}')

    value_counts = {len(group) for group in groups}
    if len(value_counts) != 1:
        raise ValueError(f'{path}: inconsistent group sizes: {[len(group) for group in groups]}')

    return model_names, groups


def infer_ratios(groups):
    if not groups:
        return []
    count = len(groups[0])
    if count == 3:
        return [1, 2, 3]
    if count == 4:
        return [0, 1, 2, 3]
    return list(range(count))


In [ ]:
def padded_ylim(groups, fixed_ylim=None):
    if fixed_ylim is not None:
        return fixed_ylim

    values = [value for group in groups for value in group]
    ymin = min(values)
    ymax = max(values)
    padding = max((ymax - ymin) * 0.08, 0.25)
    return ymin - padding, ymax + padding


def legend_location(ylabel):
    if ylabel == 'WSR (%)':
        return 'lower right'
    return 'center right'


def draw_metric(data_file, output_file, ylabel, fixed_ylim=None, y_ticks=None):
    data_path = BASE_DIR / data_file
    output_path = BASE_DIR / output_file
    model_names, groups = load_metric_file(data_path)
    if not groups:
        print(f'Skip {data_file}: empty file')
        return None

    ratios = infer_ratios(groups)
    plt.rcParams.update(
        {
            'font.family': 'serif',
            'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
            'axes.linewidth': 1.1,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'axes.labelsize': 13,
            'legend.fontsize': 8.5,
        }
    )

    fig, ax = plt.subplots(figsize=(4.2, 3.0), dpi=300)
    for model, values, color, marker in zip(model_names, groups, COLORS, MARKERS):
        ax.plot(
            ratios,
            values,
            label=model,
            color=color,
            linewidth=1.55,
            marker=marker,
            markersize=4.8,
            markerfacecolor='white',
            markeredgewidth=1.25,
        )

    ax.set_xlabel('Marking Ratio (%)')
    ax.set_ylabel(ylabel)
    ax.set_xticks(ratios)
    ax.set_xticklabels([str(x) for x in ratios])
    ax.set_xlim(min(ratios) - 0.18, max(ratios) + 0.18)
    ax.set_ylim(*padded_ylim(groups, fixed_ylim=fixed_ylim))
    if y_ticks is not None:
        ax.set_yticks(y_ticks)
    ax.grid(True, axis='both', linestyle=(0, (6, 6)), color='#bfbfbf', linewidth=0.75)
    ax.legend(
        loc=legend_location(ylabel),
        frameon=True,
        framealpha=0.92,
        edgecolor='black',
        handlelength=1.9,
        borderpad=0.35,
        labelspacing=0.25,
    )
    fig.tight_layout(pad=0.35)
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0.03)
    plt.close(fig)
    print(output_path)
    return output_path


In [ ]:
outputs = []
outputs.append(
    draw_metric(
        'wsr.txt',
        'wsr_opmark_ref_123_line.png',
        'WSR (%)',
        fixed_ylim=(89.5, 100.5),
        y_ticks=[90, 92, 94, 96, 98, 100],
    )
)
outputs.append(draw_metric('bleu.txt', 'bleu_opmark_ref_123_line.png', 'BLEU (%)'))
outputs.append(draw_metric('codebleu.txt', 'codebleu_opmark_ref_123_line.png', 'CodeBLEU (%)'))
[output for output in outputs if output is not None]
